## 2. 감성 사전 기반 분석 (Lexicon-based)

미리 정의된 감성 단어 사전을 사용하여 텍스트의 전체 감성 점수를 계산하는 방식입니다.

### 2.1 알고리즘 동작 원리 (소규모 데이터 증명)

**가정: 간단한 감성 사전**
- Positive: `great` (+2), `happy` (+3)
- Negative: `bad` (-2), `disappointing` (-3)

**시나리오 데이터**
- 문장 A: "The movie was **great** and I am **happy**."
- 문장 B: "The plot was **bad** and **disappointing**."

**증명 계산**
1. **문장 A 점수**: $Score(\text{"great"}) + Score(\text{"happy"}) = 2 + 3 = +5$ (긍정)
2. **문장 B 점수**: $Score(\text{"bad"}) + Score(\text{"disappointing"}) = (-2) + (-3) = -5$ (부정)

### 2.2 주요 감성 분석 도구별 상세 설명

사전 기반 분석을 지원하는 각 도구의 유래와 동작 특징은 다음과 같습니다.

#### 1) TextBlob
- **개요**: NLTK와 Pattern 라이브러리를 기반으로 하며, 직관적인 API를 제공하는 범용 NLP 라이브러리입니다.
- **특징**: 단순히 단어 점수를 더하는 것이 아니라, 문장 내 단어들의 상관관계(부정어, 강조어)를 파악하여 **평균 극성(Polarity)** 을 산출합니다. 또한 텍스트가 얼마나 주관적인지를 나타내는 **주관성(Subjectivity)** 지수를 함께 제공하는 것이 장점입니다.

#### 2) AFINN
- **개요**: Finn Årup Nielsen에 의해 구축된 가장 단순하고 직관적인 감성 사전 중 하나입니다.
- **특징**: 단어마다 -5(매우 부정)부터 +5(매우 긍정)까지의 **정수 점수** 가 매겨져 있습니다. 복잡한 문법 규칙 없이 텍스트에 포함된 단어 점수를 모두 더하는 방식이기 때문에 계산 속도가 매우 빠르고 성능이 안정적입니다.

#### 3) VADER (Valence Aware Dictionary and sEntiment Reasoner)
- **개요**: 소셜 미디어(SNS) 텍스트의 감성 분석을 위해 특별히 설계된 규칙 기반 모델입니다.
- **특징**: 감성 사전뿐만 아니라 **문법적 휴리스틱(Heuristics)** 을 사용합니다. 예를 들어, "GREAT"처럼 대문자로 쓰거나 "good!!!"처럼 느낌표를 사용하는 경우 감성 강도를 높게 평가합니다. 또한 "but"과 같은 접속사 뒤에 오는 문장에 더 높은 가중치를 두는 등 문맥 파악 능력이 뛰어납니다.

### 2.3 도구 간 알고리즘 및 활용 비교

| 특징 | **TextBlob** | **AFINN** | **VADER** |
| :--- | :--- | :--- | :--- |
| **기본 알고리즘** | 패턴 기반 (사전 + 규칙) | 단순 사전 합산 | 규칙 기반 (Social Media 특화) |
| **점수 산출 방식** | **평균 점수** (-1.0 ~ 1.0) | **총합 점수** (정수 위주) | **복합 점수** (Compound Score) |
| **부정어/강조어** | 지원 (매우 강력함) | 미지원 (단어별 독립) | 지원 (문맥 및 문장부호 포함) |
| **특이 사항** | 주관성(Subjectivity) 지수 제공 | 가장 빠르고 직관적임 | 대문자, 구두점, 이모티콘 반영 |
| **주요 활용** | 일반적인 문장 분석 | 빠른 키워드 기반 분석 | 소셜 미디어(SNS), 댓글 분석 |

#### 핵심 차이점 요약
1. **계산 방식**: TextBlob은 문장의 감성 단어 개수로 나누어 **밀도(평균)** 를 측정하는 반면, AFINN은 단순히 단어들의 가중치를 **누적**합니다.
2. **문맥 이해**: VADER는 "GOOD!!!"과 "good"의 차이를 인식하며, "but"과 같은 접속사가 감성 흐름을 바꾸는 것까지 고려하는 정교한 규칙을 가집니다.
3. **데이터 특성**: 분석하고자 하는 텍스트가 정제된 글이라면 TextBlob이 유리하고, 비격식적인 SNS 데이터라면 VADER가 가장 높은 성능을 보입니다.

### 2.4 TextBlob 알고리즘 상세 설명

TextBlob은 단순히 단어 점수를 더하는 것을 넘어, **패턴 기반 사후 처리(Pattern-based processing)** 를 수행합니다.

#### 동작 원리
1. **사전 매핑**: 문장 내 단어들을 사전에 정의된 **Polarity(극성)** 및 **Subjectivity(주관성)** 값과 매핑합니다.
2. **부정어 처리 (Negation)**: 'not', 'never' 등이 감성 단어 앞에 등장할 경우, 극성 값에 특정 가중치(보통 -0.5)를 곱하여 의미를 반전시킵니다.
3. **강조어 처리 (Intensity)**: 'very', 'really' 등은 뒤따르는 단어의 극성 절대값을 증폭시킵니다.
4. **최종 평균 산출**: 문장에서 발견된 모든 감성 단어의 수정된 점수들을 합산한 후, 단어 개수로 나누어 최종 극성값을 결정합니다.

### 한국어 확장

1. **형태소 분석 및 원형 복원**: '좋아요', '좋다', '좋네요' 등을 하나의 감성 단어 '좋다'로 인식하기 위해 형태소 분석기(Okt 등)를 통한 어간 추출(Stemming)이 필수적입니다.
2. **한국어 전용 사전 구축**: '최고', '나쁘다' 등 한국어 감성어에 대한 극성값을 정의합니다.
3. **한국어 문법 규칙 적용**: '안 좋다', '좋지 않다'와 같이 한국어 특유의 부정 표현 방식을 반영하여 점수를 반전시키는 로직을 구현합니다.